# Analysis of constant acceleration MPC, for the C++ implementation

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
import jax.numpy as jnp
import numpy as np
import tqdm
import h5py

import matplotlib.pyplot as plt

import exp_mpc.stewart_min.robo as robo
import exp_mpc.stewart_min.vest as vest
import exp_mpc.stewart_min.utils as utils
import exp_mpc.stewart_min.viz as viz

In [ ]:
# general setup

params = robo.RoboParams()
geom = robo.RoboGeom()

T = 20.0  # s
num_steps = int(T / params.dt)
n = 200  # horizon

acc_ref = jnp.array([1.0, 0.0, 0.0]) + robo.gravity
acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
omega_ref = jnp.array([0.0, 0.0, 0.1])
omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

vspec_acc = vest.spec_refs["acc0"][1]
vspec_omega = vest.spec_refs["omega0"][1]

## visualize results

In [ ]:
@jax.jit
def table_sol_from_flat_control(
    sol0: utils.TableSol,  # last solution
    flat_control: jax.Array,
    time: jax.Array,
) -> utils.TableSol:
    control = utils.Control.from_flat(flat_control)

    control0 = sol0.u.control[0]
    rstate0 = sol0.x.state[0]
    x0 = rstate0[:6]
    v0 = rstate0[6:]
    rstate1 = jnp.empty_like(rstate0)
    rstate1 = rstate1.at[:6].set(x0 + v0 * params.dt + 0.5 * control0 * params.dt**2)
    rstate1 = rstate1.at[6:].set(v0 + control0 * params.dt)

    rstate = utils.get_rstate(params.dt, control, rstate1)
    vstate_irl = utils.get_vstate_irl(vspec_acc, vspec_omega, rstate, control, sol0.u.control[0], sol0.vstate_irl.x_state[0])  # NOT off-by-one
    vstate_sim = utils.get_vstate(vspec_acc, vspec_omega, acc_ref, omega_ref, sol0.vstate_sim.x_state[1])
    table_sol = utils.TableSol(
        x=rstate,
        u=control,
        vstate_irl=vstate_irl,
        vstate_sim=vstate_sim,
        stats=utils.TableStats(time=time, status=jnp.array(0), cost=jnp.array(0.0)),
    )
    return table_sol

In [ ]:
with h5py.File("../cpp/data/mpc_example_data.h5", "r") as f:
    control_hist = np.array(f["control_hist"])
    timings  = np.array(f["timings"])

zero_control = utils.Control.from_flat(control_hist[0])
zero_rstate = utils.get_rstate(params.dt, zero_control, jnp.zeros(12, dtype=float))
vstate0_irl = jnp.zeros(15)
vstate0_sim = jnp.zeros(15)
vstate0_irl = vstate0_irl.at[4:6].set(vspec_acc.v0_earth)
vstate0_sim = vstate0_sim.at[4:6].set(vspec_acc.v0_earth)

sol0 = utils.TableSol(
    x=zero_rstate,
    u=zero_control,
    vstate_irl=utils.get_vstate_irl(vspec_acc, vspec_omega, zero_rstate, zero_control, jnp.zeros(6), vstate0_irl),
    vstate_sim=utils.get_vstate(vspec_acc, vspec_omega, acc_ref, omega_ref, vstate0_sim),
    stats=utils.TableStats(jnp.array(0.0), jnp.array(0), jnp.array(0.0)),
)

sol_list = [sol0]
times = []
for i in tqdm.tqdm(range(1, num_steps)):
    sol = table_sol_from_flat_control(sol_list[-1], control_hist[i], timings[i])
    sol_list.append(sol)
    times.append(timings[i] * 1e-6)

In [ ]:
freqs = 1.0 / np.array(times)
print(f"{float(np.min(freqs)):.2f}, {float(np.max(freqs)):.2f}, {float(np.mean(freqs)):.2f}, {float(np.std(freqs)):.2f}")

In [ ]:
plt.close("all")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(freqs)
ax.set_title("Optimization Frequency Over Time")
ax.set_xlabel("Iteration")
ax.set_ylabel("Frequency (Hz)")
ax.grid(True)
plt.show()

In [ ]:
sol_list_end = []
extra_steps = 0

# sol_list_end = viz.split_tablesol(sol_list[-1])
# extra_steps = sol_list[-1].u.shape[0] - 1

trajectory = sol_list + sol_list_end
references= {
    "xyz-acceleration": jnp.tile(A=acc_ref[0], reps=(len(trajectory), 1)),
    "angular-velocity": jnp.tile(A=omega_ref[0], reps=(len(trajectory), 1)),
}

In [ ]:
mpc_human_fig = viz.plot_human_trajectory(trajectory=trajectory, references=references, robo_params=params)

In [ ]:
mpc_vestibular_fig = viz.plot_vestibular_trajectory(trajectory=trajectory, robo_params=params)

In [ ]:
mpc_table_fig = viz.plot_cartesian_table_trajectory(trajectory=trajectory, robo_params=params)

In [ ]:
mpc_actuator_fig = viz.plot_actuator_trajectory(trajectory=trajectory, robo_params=params, robo_geom=geom)

## visualize timings

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ax.plot(np.array(times) * 1e3)
fig.suptitle("C++ timings")
ax.set_xlabel("iter")
ax.set_ylabel("time (ms)")
ax.set_ylim(2, 5)
ax.grid()
plt.show()

## animation

(WARNING: can take a long time.
Usually about as long as the video being generated.)

In [ ]:
# mp_mpl.call_mp_animate_trajectory(
#     file_name="data/cpp_const_acc_3d.mp4",
#     trajectory=trajectory,
# )